In [1]:
import pandas as pd
import numpy as np

sales = pd.read_csv("./data/sales_train.csv")
inventory = pd.read_csv("./data/inventory.csv")
calendar = pd.read_csv("./data/calendar.csv")

print("Sales shape:", sales.shape)
print("Inventory shape:", inventory.shape)
print("Calendar shape:", calendar.shape)

print("\nSales columns:")
print(sales.columns.tolist())

print("\nSales sample:")
display(sales.head())

print("\nData types:")
print(sales.dtypes)

Sales shape: (4007419, 14)
Inventory shape: (5432, 8)
Calendar shape: (23016, 7)

Sales columns:
['unique_id', 'date', 'warehouse', 'total_orders', 'sales', 'sell_price_main', 'availability', 'type_0_discount', 'type_1_discount', 'type_2_discount', 'type_3_discount', 'type_4_discount', 'type_5_discount', 'type_6_discount']

Sales sample:


,unique_id,date,warehouse,total_orders,sales,sell_price_main,availability,type_0_discount,type_1_discount,type_2_discount,type_3_discount,type_4_discount,type_5_discount,type_6_discount
0,4845,2024-03-10,Budapest_1,6436.0,16.34,646.26,1.00,0.00000,0.0,0.0,0.0,0.15312,0.0,0.0
1,4845,2021-05-25,Budapest_1,4663.0,12.63,455.96,1.00,0.00000,0.0,0.0,0.0,0.15025,0.0,0.0
2,4845,2021-12-20,Budapest_1,6507.0,34.55,455.96,1.00,0.00000,0.0,0.0,0.0,0.15025,0.0,0.0
3,4845,2023-04-29,Budapest_1,5463.0,34.52,646.26,0.96,0.20024,0.0,0.0,0.0,0.15312,0.0,0.0
4,4845,2022-04-01,Budapest_1,5997.0,35.92,486.41,1.00,0.00000,0.0,0.0,0.0,0.15649,0.0,0.0



Data types:
unique_id            int64
date                object
warehouse           object
total_orders       float64
sales              float64
sell_price_main    float64
availability       float64
type_0_discount    float64
type_1_discount    float64
type_2_discount    float64
type_3_discount    float64
type_4_discount    float64
type_5_discount    float64
type_6_discount    float64
dtype: object


In [2]:
# Missing values
missing = sales.isnull().sum()
print("Missing values:")
print(missing[missing > 0].sort_values(ascending=False))

# Duplicate rows
print("Exact duplicate rows:", sales.duplicated().sum())

# Duplicate product-warehouse-date records
key = ["unique_id", "date"]

print(
    "Duplicate product-date records:",
    sales.duplicated(subset=key).sum()
)

Missing values:
total_orders    52
sales           52
dtype: int64
Exact duplicate rows: 0
Duplicate product-date records: 0


In [3]:
discount_cols = [
    c for c in sales.columns
    if "discount" in c.lower()
]

print("Discount columns:", discount_cols)

cols_to_check = ["sell_price_main"] + discount_cols
cols_to_check = [c for c in cols_to_check if c in sales.columns]

print(sales[cols_to_check].describe().T)

for col in cols_to_check:
    print(f"\n{col}")
    print("Missing:", sales[col].isna().sum())
    print("Negative:", (sales[col] < 0).sum())
    print("Zero:", (sales[col] == 0).sum())

Discount columns: ['type_0_discount', 'type_1_discount', 'type_2_discount', 'type_3_discount', 'type_4_discount', 'type_5_discount', 'type_6_discount']
                     count        mean         std       min    25%    50%  \
sell_price_main  4007419.0  181.419909  461.101407   0.02000  21.13  45.64   
type_0_discount  4007419.0    0.008519    0.051470 -20.94930   0.00   0.00   
type_1_discount  4007419.0    0.000124    0.005143   0.00000   0.00   0.00   
type_2_discount  4007419.0    0.001163    0.018800   0.00000   0.00   0.00   
type_3_discount  4007419.0    0.000005    0.000892   0.00000   0.00   0.00   
type_4_discount  4007419.0    0.005680    0.028930  -0.04712   0.00   0.00   
type_5_discount  4007419.0    0.000405    0.009200   0.00000   0.00   0.00   
type_6_discount  4007419.0    0.010455    0.055450  -0.80149   0.00   0.00   

                    75%          max  
sell_price_main  112.13  21682.99000  
type_0_discount    0.00      0.99164  
type_1_discount    0.00     

In [4]:
sales["date"] = pd.to_datetime(sales["date"])

print(
    "Distinct selling prices:",
    sales["sell_price_main"].nunique()
)

print(
    "Rows with at least one nonzero discount:",
    (sales[discount_cols].fillna(0) != 0).any(axis=1).sum()
)

Distinct selling prices: 35723
Rows with at least one nonzero discount: 585519


In [5]:
print(sales["sales"].describe())

print("Negative sales:", (sales["sales"] < 0).sum())
print("Zero sales:", (sales["sales"] == 0).sum())
print("Missing sales:", sales["sales"].isna().sum())

print("\nAvailability values:")
print(sales["availability"].value_counts(dropna=False).head(20))

count    4.007367e+06
mean     1.083808e+02
std      3.603327e+02
min      0.000000e+00
25%      1.798000e+01
50%      3.952000e+01
75%      9.447000e+01
max      2.631619e+04
Name: sales, dtype: float64
Negative sales: 0
Zero sales: 49207
Missing sales: 52

Availability values:
availability
1.00    3035211
0.97      40419
0.98      39833
0.96      36086
0.94      26253
0.95      25048
0.93      23467
0.99      20844
0.92      19608
0.91      19411
0.87      17729
0.88      17210
0.86      17152
0.90      17048
0.89      16753
0.81      14850
0.84      14617
0.82      14567
0.85      14528
0.83      14414
Name: count, dtype: int64


In [6]:
# Check uniqueness before joining
print(
    "Duplicate inventory unique_ids:",
    inventory["unique_id"].duplicated().sum()
)

print(
    "Duplicate calendar dates:",
    calendar["date"].duplicated().sum()
)

# Check how many sales records match inventory metadata
inventory_ids = set(inventory["unique_id"].dropna())

print(
    "Sales rows without inventory match:",
    (~sales["unique_id"].isin(inventory_ids)).sum()
)

Duplicate inventory unique_ids: 0
Duplicate calendar dates: 19728
Sales rows without inventory match: 0


In [7]:
sales["date"] = pd.to_datetime(sales["date"], errors="coerce")

# 1. Missing values
print("=== MISSING VALUES ===")
print(sales.isna().sum()[sales.isna().sum() > 0])

# 2. Duplicate observations
print("\n=== DUPLICATES ===")
print("Exact duplicate rows:", sales.duplicated().sum())
print(
    "Duplicate product-date rows:",
    sales.duplicated(subset=["unique_id", "date"]).sum()
)

# 3. Invalid or suspicious values
print("\n=== VALUE CHECKS ===")
print("Invalid dates:", sales["date"].isna().sum())
print("Sales below zero:", (sales["sales"] < 0).sum())
print("Sales equal to zero:", (sales["sales"] == 0).sum())
print("Prices <= 0:", (sales["sell_price_main"] <= 0).sum())

# 4. Time coverage
print("\n=== DATE COVERAGE ===")
print("Start:", sales["date"].min())
print("End:", sales["date"].max())
print("Unique dates:", sales["date"].nunique())

# 5. Product and warehouse coverage
print("\n=== SERIES COVERAGE ===")
print("Unique series:", sales["unique_id"].nunique())
print("Warehouses:", sales["warehouse"].nunique())
print(sales["warehouse"].value_counts())


=== MISSING VALUES ===
total_orders    52
sales           52
dtype: int64

=== DUPLICATES ===
Exact duplicate rows: 0
Duplicate product-date rows: 0

=== VALUE CHECKS ===
Invalid dates: 0
Sales below zero: 0
Sales equal to zero: 49207
Prices <= 0: 0

=== DATE COVERAGE ===
Start: 2020-08-01 00:00:00
End: 2024-06-02 00:00:00
Unique dates: 1402

=== SERIES COVERAGE ===
Unique series: 5390
Warehouses: 7
warehouse
Prague_1       780566
Prague_3       779655
Prague_2       770709
Brno_1         643637
Budapest_1     574582
Munich_1       259333
Frankfurt_1    198937
Name: count, dtype: int64


In [8]:
discount_cols = [
    f"type_{i}_discount" for i in range(7)
]

print("=== PRICE SUMMARY ===")
print(sales["sell_price_main"].describe())

print("\n=== DISCOUNT SUMMARY ===")
print(sales[discount_cols].describe().T)

print("\n=== NONZERO DISCOUNT OBSERVATIONS ===")
for col in discount_cols:
    print(
        f"{col}:",
        int((sales[col].fillna(0) != 0).sum()),
        "nonzero rows"
    )

# Investigate unusual negative discount values
print("\n=== NEGATIVE DISCOUNT VALUES ===")
for col in discount_cols:
    count = int((sales[col] < 0).sum())
    if count:
        print(f"\n{col}: {count} negative values")
        display(
            sales.loc[sales[col] < 0,
                      ["unique_id", "date", "sell_price_main", col]]
            .head(10)
        )

# Check whether multiple discount fields are active together
active_discounts = (
    sales[discount_cols].fillna(0).ne(0).sum(axis=1)
)

print(
    "\nRows with multiple nonzero discount fields:",
    int((active_discounts > 1).sum())
)


=== PRICE SUMMARY ===
count    4.007419e+06
mean     1.814199e+02
std      4.611014e+02
min      2.000000e-02
25%      2.113000e+01
50%      4.564000e+01
75%      1.121300e+02
max      2.168299e+04
Name: sell_price_main, dtype: float64

=== DISCOUNT SUMMARY ===
                     count      mean       std       min  25%  50%  75%  \
type_0_discount  4007419.0  0.008519  0.051470 -20.94930  0.0  0.0  0.0   
type_1_discount  4007419.0  0.000124  0.005143   0.00000  0.0  0.0  0.0   
type_2_discount  4007419.0  0.001163  0.018800   0.00000  0.0  0.0  0.0   
type_3_discount  4007419.0  0.000005  0.000892   0.00000  0.0  0.0  0.0   
type_4_discount  4007419.0  0.005680  0.028930  -0.04712  0.0  0.0  0.0   
type_5_discount  4007419.0  0.000405  0.009200   0.00000  0.0  0.0  0.0   
type_6_discount  4007419.0  0.010455  0.055450  -0.80149  0.0  0.0  0.0   

                     max  
type_0_discount  0.99164  
type_1_discount  0.50000  
type_2_discount  0.76372  
type_3_discount  0.25063  
ty

,unique_id,date,sell_price_main,type_0_discount
668474,1123,2022-01-19,607.43,-0.42066
668581,1123,2022-01-20,607.43,-0.20402
2250867,2402,2021-09-14,1065.89,-0.04032
2251021,2402,2021-09-12,1065.89,-0.04032
2251031,2402,2021-09-08,1065.89,-0.04032
2251112,2402,2021-09-29,1065.89,-0.04032
2251176,2402,2021-10-04,1065.89,-0.04032
2251480,2402,2021-09-11,1065.89,-0.04032
2251540,2402,2021-09-30,1065.89,-0.04032
2251547,2402,2021-10-03,1065.89,-0.04032



type_4_discount: 1 negative values


,unique_id,date,sell_price_main,type_4_discount
211,4845,2021-05-20,455.96,-0.04712



type_6_discount: 4 negative values


,unique_id,date,sell_price_main,type_6_discount
2789972,4447,2022-03-11,129.13,-0.20941
2790605,4447,2022-03-10,129.20,-0.80149
2948477,5303,2022-02-01,7640.56,-0.53746
2949020,5303,2022-01-31,7513.00,-0.31571



Rows with multiple nonzero discount fields: 19714


In [9]:
print("=== TOP-LEVEL CATEGORIES ===")
print(inventory["L1_category_name_en"].value_counts())

print("\n=== PRODUCT METADATA ===")
print("Inventory rows:", len(inventory))
print("Unique series:", inventory["unique_id"].nunique())
print("Unique products:", inventory["product_unique_id"].nunique())

# Check inventory metadata matches the sales table
inventory_ids = set(inventory["unique_id"])
sales_ids = set(sales["unique_id"])

print("\n=== JOIN VALIDATION ===")
print("Sales series without metadata:", len(sales_ids - inventory_ids))
print("Metadata series without sales:", len(inventory_ids - sales_ids))

# Inspect product names and category hierarchy
display(
    inventory[
        [
            "unique_id",
            "product_unique_id",
            "name",
            "L1_category_name_en",
            "L2_category_name_en"
        ]
    ].head(15)
)


=== TOP-LEVEL CATEGORIES ===
L1_category_name_en
Fruit and vegetable    3282
Bakery                 1616
Meat and fish           534
Name: count, dtype: int64

=== PRODUCT METADATA ===
Inventory rows: 5432
Unique series: 5432
Unique products: 2670

=== JOIN VALIDATION ===
Sales series without metadata: 0
Metadata series without sales: 42


,unique_id,product_unique_id,name,L1_category_name_en,L2_category_name_en
0,5255,2583,Pastry_196,Bakery,Bakery_L2_14
1,4948,2426,Herb_19,Fruit and vegetable,Fruit and vegetable_L2_30
2,2146,1079,Beet_2,Fruit and vegetable,Fruit and vegetable_L2_3
3,501,260,Chicken_13,Meat and fish,Meat and fish_L2_13
4,4461,2197,Chicory_1,Fruit and vegetable,Fruit and vegetable_L2_17
5,4119,2024,Donut_32,Bakery,Bakery_L2_18
6,1281,653,Salmon_3,Meat and fish,Meat and fish_L2_25
7,4975,2439,Chicken_93,Meat and fish,Meat and fish_L2_13
8,5032,2465,Physalis_2,Fruit and vegetable,Fruit and vegetable_L2_1
9,240,120,Grape_2,Fruit and vegetable,Fruit and vegetable_L2_1


In [10]:
discount_cols = [f"type_{i}_discount" for i in range(7)]

# A. Inspect rows with missing sales or order counts
missing_rows = sales[
    sales["sales"].isna() | sales["total_orders"].isna()
]

print("Rows with missing sales/order information:", len(missing_rows))
display(missing_rows.head(10))

print("\nMissingness overlap:")
print(pd.crosstab(
    sales["sales"].isna(),
    sales["total_orders"].isna(),
    rownames=["sales_missing"],
    colnames=["orders_missing"]
))

# B. Inspect discount values outside the conventional [0, 1] range
for col in discount_cols:
    suspicious = sales[
        (sales[col] < 0) | (sales[col] > 1)
    ]

    if not suspicious.empty:
        print(f"\n{col}: {len(suspicious)} suspicious rows")
        display(suspicious[
            ["unique_id", "date", "warehouse",
             "sales", "sell_price_main", col]
        ].head(10))

# C. Calendar preparation
calendar["date"] = pd.to_datetime(
    calendar["date"], errors="coerce"
)

print("\nDuplicate calendar date-warehouse keys:",
      calendar.duplicated(["date", "warehouse"]).sum())

# A duplicate date alone is expected across warehouses.
# A duplicate date-WAREHOUSE key is what we need to check.

# D. Join calendar using BOTH date and warehouse
calendar_features = [
    "date", "warehouse", "holiday", "shops_closed",
    "winter_school_holidays", "school_holidays"
]

calendar_small = calendar[calendar_features].copy()

model_data = sales.merge(
    inventory.drop(columns=["warehouse"]),
    on="unique_id",
    how="left",
    validate="many_to_one",
    indicator="inventory_merge"
)

model_data = model_data.merge(
    calendar_small,
    on=["date", "warehouse"],
    how="left",
    validate="many_to_one",
    indicator="calendar_merge"
)

print("\nFinal merge checks:")
print(model_data["inventory_merge"].value_counts())
print(model_data["calendar_merge"].value_counts())
print("Rows before joins:", len(sales))
print("Rows after joins:", len(model_data))


Rows with missing sales/order information: 52


,unique_id,date,warehouse,total_orders,sales,sell_price_main,availability,type_0_discount,type_1_discount,type_2_discount,type_3_discount,type_4_discount,type_5_discount,type_6_discount
154017,885,2021-05-23,Munich_1,NaN,NaN,4.02,1.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0
154099,885,2021-05-21,Munich_1,NaN,NaN,4.02,1.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0
154149,885,2021-05-24,Munich_1,NaN,NaN,4.02,1.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0
154155,885,2021-05-22,Munich_1,NaN,NaN,4.02,1.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0
429815,1237,2021-12-10,Frankfurt_1,NaN,NaN,3.84,0.3,0.0000,0.0,0.0,0.0,0.0,0.0,0.0
429860,1237,2021-12-09,Frankfurt_1,NaN,NaN,3.84,1.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0
733468,725,2021-06-27,Munich_1,NaN,NaN,2.82,1.0,0.1000,0.0,0.0,0.0,0.0,0.0,0.0
734198,725,2021-06-26,Munich_1,NaN,NaN,2.82,1.0,0.1000,0.0,0.0,0.0,0.0,0.0,0.0
738594,3778,2021-05-29,Munich_1,NaN,NaN,2.75,1.0,0.2446,0.0,0.0,0.0,0.0,0.0,0.0
738600,3778,2021-05-23,Munich_1,NaN,NaN,2.75,1.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0



Missingness overlap:
orders_missing    False  True 
sales_missing                 
False           4007367      0
True                  0     52

type_0_discount: 27 suspicious rows


,unique_id,date,warehouse,sales,sell_price_main,type_0_discount
668474,1123,2022-01-19,Budapest_1,281.64,607.43,-0.42066
668581,1123,2022-01-20,Budapest_1,232.73,607.43,-0.20402
2250867,2402,2021-09-14,Budapest_1,24.61,1065.89,-0.04032
2251021,2402,2021-09-12,Budapest_1,19.29,1065.89,-0.04032
2251031,2402,2021-09-08,Budapest_1,41.54,1065.89,-0.04032
2251112,2402,2021-09-29,Budapest_1,21.93,1065.89,-0.04032
2251176,2402,2021-10-04,Budapest_1,34.69,1065.89,-0.04032
2251480,2402,2021-09-11,Budapest_1,29.75,1065.89,-0.04032
2251540,2402,2021-09-30,Budapest_1,46.68,1065.89,-0.04032
2251547,2402,2021-10-03,Budapest_1,35.37,1065.89,-0.04032



type_4_discount: 1 suspicious rows


,unique_id,date,warehouse,sales,sell_price_main,type_4_discount
211,4845,2021-05-20,Budapest_1,11.01,455.96,-0.04712



type_6_discount: 4 suspicious rows


,unique_id,date,warehouse,sales,sell_price_main,type_6_discount
2789972,4447,2022-03-11,Prague_2,14.25,129.13,-0.20941
2790605,4447,2022-03-10,Prague_2,3.27,129.20,-0.80149
2948477,5303,2022-02-01,Budapest_1,3.44,7640.56,-0.53746
2949020,5303,2022-01-31,Budapest_1,0.63,7513.00,-0.31571



Duplicate calendar date-warehouse keys: 0

Final merge checks:
inventory_merge
both          4007419
left_only           0
right_only          0
Name: count, dtype: int64
calendar_merge
both          4007419
left_only           0
right_only          0
Name: count, dtype: int64
Rows before joins: 4007419
Rows after joins: 4007419


In [11]:
discount_cols = [f"type_{i}_discount" for i in range(7)]

# Replace negative discount values with zero
sales[discount_cols] = sales[discount_cols].clip(lower=0)

# Verify the transformation
print(sales[discount_cols].min())

type_0_discount    0.0
type_1_discount    0.0
type_2_discount    0.0
type_3_discount    0.0
type_4_discount    0.0
type_5_discount    0.0
type_6_discount    0.0
dtype: float64


In [12]:
model_data[discount_cols] = model_data[discount_cols].clip(lower=0)

print("Minimum discount values after correction:")
print(model_data[discount_cols].min())

Minimum discount values after correction:
type_0_discount    0.0
type_1_discount    0.0
type_2_discount    0.0
type_3_discount    0.0
type_4_discount    0.0
type_5_discount    0.0
type_6_discount    0.0
dtype: float64


In [13]:
discount_cols = [f"type_{i}_discount" for i in range(7)]

# Keep the original data unchanged
model_data = sales.copy()

# Inspect before cleaning
print(model_data[discount_cols].describe().T)
print("Missing values:")
print(model_data[discount_cols].isna().sum())
print("Negative values:")
print((model_data[discount_cols] < 0).sum())

# Apply the negative-discount correction
model_data[discount_cols] = model_data[discount_cols].clip(lower=0)

# Check how much variation each column has
print("Distinct values:")
print(model_data[discount_cols].nunique().sort_values(ascending=False))

                     count      mean       std  min  25%  50%  75%      max
type_0_discount  4007419.0  0.008533  0.048938  0.0  0.0  0.0  0.0  0.99164
type_1_discount  4007419.0  0.000124  0.005143  0.0  0.0  0.0  0.0  0.50000
type_2_discount  4007419.0  0.001163  0.018800  0.0  0.0  0.0  0.0  0.76372
type_3_discount  4007419.0  0.000005  0.000892  0.0  0.0  0.0  0.0  0.25063
type_4_discount  4007419.0  0.005680  0.028930  0.0  0.0  0.0  0.0  0.45290
type_5_discount  4007419.0  0.000405  0.009200  0.0  0.0  0.0  0.0  0.66914
type_6_discount  4007419.0  0.010455  0.055448  0.0  0.0  0.0  0.0  0.90005
Missing values:
type_0_discount    0
type_1_discount    0
type_2_discount    0
type_3_discount    0
type_4_discount    0
type_5_discount    0
type_6_discount    0
dtype: int64
Negative values:
type_0_discount    0
type_1_discount    0
type_2_discount    0
type_3_discount    0
type_4_discount    0
type_5_discount    0
type_6_discount    0
dtype: int64
Distinct values:
type_6_discount    446

In [14]:
df = model_data.dropna(subset=["sales"]).copy()

discount_col = "type_0_discount"

# Use quantile bins to avoid creating many nearly empty groups
df["discount_bin"] = pd.qcut(
    df[discount_col],
    q=5,
    duplicates="drop"
)

discount_summary = (
    df.groupby("discount_bin", observed=True)
      .agg(
          observations=("sales", "size"),
          average_sales=("sales", "mean"),
          median_sales=("sales", "median"),
          average_price=("sell_price_main", "mean"),
          average_availability=("availability", "mean")
      )
      .reset_index()
)

print(discount_summary)

      discount_bin  observations  average_sales  median_sales  average_price  \
0  (-0.001, 0.992]       4007367     108.380753         39.52     181.422218   

   average_availability  
0              0.925311  


In [15]:
discount_cols = [f"type_{i}_discount" for i in range(7)]

# Count zero values and calculate percentages
zero_discount_summary = pd.DataFrame({
    "Zero_Count": (sales[discount_cols] == 0).sum(),
    "Non_Zero_Count": (sales[discount_cols] != 0).sum(),
    "Zero_Percentage": (sales[discount_cols] == 0).mean() * 100
})

# Display results
print(zero_discount_summary.round(2))

                 Zero_Count  Non_Zero_Count  Zero_Percentage
type_0_discount     3862139          145280            96.37
type_1_discount     4004907            2512            99.94
type_2_discount     3988398           19021            99.53
type_3_discount     4007240             179           100.00
type_4_discount     3856907          150512            96.24
type_5_discount     3998389            9030            99.77
type_6_discount     3728625          278794            93.04


In [16]:
print("Sales shape:", sales.shape)
print("Inventory shape:", inventory.shape)

print("\nSales columns:")
print(sales.columns.tolist())

print("\nInventory columns:")
print(inventory.columns.tolist())

print("\nDuplicate inventory unique_id values:",
      inventory["unique_id"].duplicated().sum())

print("\nDuplicate sales unique_id-date pairs:",
      sales.duplicated(["unique_id", "date"]).sum())

Sales shape: (4007419, 14)
Inventory shape: (5432, 8)

Sales columns:
['unique_id', 'date', 'warehouse', 'total_orders', 'sales', 'sell_price_main', 'availability', 'type_0_discount', 'type_1_discount', 'type_2_discount', 'type_3_discount', 'type_4_discount', 'type_5_discount', 'type_6_discount']

Inventory columns:
['unique_id', 'product_unique_id', 'name', 'L1_category_name_en', 'L2_category_name_en', 'L3_category_name_en', 'L4_category_name_en', 'warehouse']

Duplicate inventory unique_id values: 0

Duplicate sales unique_id-date pairs: 0


In [17]:
sales_with_inventory = sales.merge(
    inventory,
    on="unique_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("Rows before merge:", len(sales))
print("Rows after merge:", len(sales_with_inventory))

print("\nMerge results:")
print(sales_with_inventory["_merge"].value_counts())

print("\nMissing product names:",
      sales_with_inventory["name"].isna().sum())

# Remove the merge diagnostic column after checking
sales_with_inventory = sales_with_inventory.drop(
    columns="_merge"
)

Rows before merge: 4007419
Rows after merge: 4007419

Merge results:
_merge
both          4007419
left_only           0
right_only          0
Name: count, dtype: int64

Missing product names: 0


In [18]:
print("Sales shape:", sales.shape)
print("Inventory shape:", inventory.shape)
print("Calendar shape:", calendar.shape)

print("\nSales memory usage (MB):")
print(round(
    sales.memory_usage(deep=True).sum() / 1024**2, 2
))

Sales shape: (4007419, 14)
Inventory shape: (5432, 8)
Calendar shape: (23016, 7)

Sales memory usage (MB):
615.74


In [19]:
import pandas as pd

# Use only product metadata columns from inventory
inventory_cols = [
    "unique_id",
    "product_unique_id",
    "name",
    "L1_category_name_en",
    "L2_category_name_en",
    "L3_category_name_en",
    "L4_category_name_en"
]

# Check that the original sales table has unique column labels
assert sales.columns.is_unique, (
    "Sales has duplicate column names. Reload the original sales CSV."
)

assert inventory["unique_id"].is_unique, (
    "Inventory has duplicate unique_id values."
)

# Merge without creating a full copy of sales first
sales_with_inventory = sales.merge(
    inventory[inventory_cols],
    on="unique_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("Rows before:", len(sales))
print("Rows after:", len(sales_with_inventory))
print(sales_with_inventory["_merge"].value_counts())

sales_with_inventory.drop(columns="_merge", inplace=True)

Rows before: 4007419
Rows after: 4007419
_merge
both          4007419
left_only           0
right_only          0
Name: count, dtype: int64


In [20]:
# Convert date columns
sales_with_inventory["date"] = pd.to_datetime(
    sales_with_inventory["date"]
)

calendar["date"] = pd.to_datetime(calendar["date"])

# Check for duplicate join keys
assert not calendar.duplicated(
    ["date", "warehouse"]
).any(), "Duplicate calendar keys found."

# Merge
model_data = sales_with_inventory.merge(
    calendar,
    on=["date", "warehouse"],
    how="left",
    validate="many_to_one",
    indicator=True
)

print("Rows before:", len(sales_with_inventory))
print("Rows after:", len(model_data))

print("\nCalendar matching:")
print(model_data["_merge"].value_counts())

model_data.drop(columns="_merge", inplace=True)

Rows before: 4007419
Rows after: 4007419

Calendar matching:
_merge
both          4007419
left_only           0
right_only          0
Name: count, dtype: int64


In [21]:
required_cols = [
    "unique_id",
    "date",
    "warehouse",
    "sales",
    "sell_price_main",
    "availability"
] + [f"type_{i}_discount" for i in range(7)]

sales_small = sales[required_cols]

print(
    "Selected data memory (MB):",
    round(
        sales_small.memory_usage(deep=True).sum() / 1024**2,
        2
    )
)

Selected data memory (MB): 585.17


In [22]:
print("Final dataset shape:", model_data.shape)

print("\nFirst 5 rows:")
display(model_data.head())

print("\nColumn names:")
print(model_data.columns.tolist())

print("\nMissing values in important columns:")
important_cols = [
    "unique_id",
    "date",
    "warehouse",
    "sales",
    "sell_price_main",
    "availability",
    "name",
    "L1_category_name_en"
]

print(model_data[important_cols].isna().sum())

Final dataset shape: (4007419, 25)

First 5 rows:


,unique_id,date,warehouse,total_orders,sales,sell_price_main,availability,type_0_discount,type_1_discount,type_2_discount,...,name,L1_category_name_en,L2_category_name_en,L3_category_name_en,L4_category_name_en,holiday_name,holiday,shops_closed,winter_school_holidays,school_holidays
0,4845,2024-03-10,Budapest_1,6436.0,16.34,646.26,1.00,0.00000,0.0,0.0,...,Croissant_35,Bakery,Bakery_L2_18,Bakery_L3_83,Bakery_L4_1,NaN,0,0,0,0
1,4845,2021-05-25,Budapest_1,4663.0,12.63,455.96,1.00,0.00000,0.0,0.0,...,Croissant_35,Bakery,Bakery_L2_18,Bakery_L3_83,Bakery_L4_1,NaN,0,0,0,0
2,4845,2021-12-20,Budapest_1,6507.0,34.55,455.96,1.00,0.00000,0.0,0.0,...,Croissant_35,Bakery,Bakery_L2_18,Bakery_L3_83,Bakery_L4_1,NaN,0,0,0,0
3,4845,2023-04-29,Budapest_1,5463.0,34.52,646.26,0.96,0.20024,0.0,0.0,...,Croissant_35,Bakery,Bakery_L2_18,Bakery_L3_83,Bakery_L4_1,NaN,0,0,0,0
4,4845,2022-04-01,Budapest_1,5997.0,35.92,486.41,1.00,0.00000,0.0,0.0,...,Croissant_35,Bakery,Bakery_L2_18,Bakery_L3_83,Bakery_L4_1,NaN,0,0,0,0



Column names:
['unique_id', 'date', 'warehouse', 'total_orders', 'sales', 'sell_price_main', 'availability', 'type_0_discount', 'type_1_discount', 'type_2_discount', 'type_3_discount', 'type_4_discount', 'type_5_discount', 'type_6_discount', 'product_unique_id', 'name', 'L1_category_name_en', 'L2_category_name_en', 'L3_category_name_en', 'L4_category_name_en', 'holiday_name', 'holiday', 'shops_closed', 'winter_school_holidays', 'school_holidays']

Missing values in important columns:
unique_id               0
date                    0
warehouse               0
sales                  52
sell_price_main         0
availability            0
name                    0
L1_category_name_en     0
dtype: int64


In [23]:
print(
    "Duplicate product-date rows:",
    model_data.duplicated(["unique_id", "date"]).sum()
)

print(
    "Unique product-warehouse series:",
    model_data["unique_id"].nunique()
)

print(
    "Date range:",
    model_data["date"].min(),
    "to",
    model_data["date"].max()
)

Duplicate product-date rows: 0
Unique product-warehouse series: 5390
Date range: 2020-08-01 00:00:00 to 2024-06-02 00:00:00


In [24]:
discount_cols = [f"type_{i}_discount" for i in range(7)]

category_discount_summary = (
    model_data
    .groupby("L1_category_name_en")[discount_cols]
    .agg(lambda x: (x > 0).mean() * 100)
    .round(2)
)

print("Percentage of observations with active discounts:")
display(category_discount_summary)

Percentage of observations with active discounts:


,type_0_discount,type_1_discount,type_2_discount,type_3_discount,type_4_discount,type_5_discount,type_6_discount
L1_category_name_en,,,,,,,
Bakery,2.03,0.06,0.36,0.00,8.08,0.18,4.87
Fruit and vegetable,4.69,0.08,0.66,0.00,1.41,0.25,3.83
Meat and fish,4.11,0.00,0.07,0.04,0.30,0.25,26.19


In [25]:

import pandas as pd

discount_cols = [f"type_{i}_discount" for i in range(7)]

# One row per item, with its category and discount statistics
item_discount_summary = (
    model_data
    .groupby(
        ["unique_id", "name", "L1_category_name_en"],
        dropna=False
    )
    .agg(
        observations=("date", "count"),
        average_price=("sell_price_main", "mean"),
        **{
            f"{col}_active_pct": (
                col, lambda x: (x > 0).mean() * 100
            )
            for col in discount_cols
        },
        **{
            f"{col}_avg_when_active": (
                col, lambda x: x[x > 0].mean()
            )
            for col in discount_cols
        }
    )
    .reset_index()
)

item_discount_summary.head(10)


,unique_id,name,L1_category_name_en,observations,average_price,type_0_discount_active_pct,type_1_discount_active_pct,type_2_discount_active_pct,type_3_discount_active_pct,type_4_discount_active_pct,type_5_discount_active_pct,type_6_discount_active_pct,type_0_discount_avg_when_active,type_1_discount_avg_when_active,type_2_discount_avg_when_active,type_3_discount_avg_when_active,type_4_discount_avg_when_active,type_5_discount_avg_when_active,type_6_discount_avg_when_active
0,0,Plum_0,Fruit and vegetable,35,773.424857,28.571429,0.0,20.0,0.0,0.000000,0.0,0.000000,0.252309,NaN,0.433986,NaN,NaN,NaN,NaN
1,1,Blueberry_0,Fruit and vegetable,189,2.783862,9.523810,0.0,0.0,0.0,15.343915,0.0,0.000000,0.161433,NaN,NaN,NaN,0.290814,NaN,NaN
2,2,Blueberry_0,Fruit and vegetable,51,3.340000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,Ginger_0,Fruit and vegetable,373,829.260107,10.187668,0.0,0.0,0.0,0.000000,0.0,0.804290,0.178057,NaN,NaN,NaN,NaN,NaN,0.411020
4,5,Onion_0,Fruit and vegetable,483,2.069172,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,6,Onion_0,Fruit and vegetable,852,1.822993,1.291080,0.0,0.0,0.0,0.000000,0.0,0.000000,0.102062,NaN,NaN,NaN,NaN,NaN,NaN
6,7,Herb_0,Fruit and vegetable,1049,1.077855,0.571973,0.0,0.0,0.0,3.145853,0.0,0.000000,0.097725,NaN,NaN,NaN,0.153850,NaN,NaN
7,8,Tomato_0,Fruit and vegetable,714,826.425756,9.663866,0.0,0.0,0.0,4.761905,0.0,1.400560,0.148718,NaN,NaN,NaN,0.095155,NaN,0.070215
8,9,Pastry_0,Bakery,1380,37.297275,0.072464,0.0,0.0,0.0,58.043478,0.0,47.101449,0.113970,NaN,NaN,NaN,0.149989,NaN,0.090206
9,10,Pastry_0,Bakery,1381,50.209783,0.072411,0.0,0.0,0.0,58.073860,0.0,50.108617,0.112100,NaN,NaN,NaN,0.149963,NaN,0.090609


In [26]:
category = "Meat and fish"  # Change this to another category
item_discount_summary[
    item_discount_summary["L1_category_name_en"] == category
].sort_values(
    "type_6_discount_active_pct",
    ascending=False
).head(20)


,unique_id,name,L1_category_name_en,observations,average_price,type_0_discount_active_pct,type_1_discount_active_pct,type_2_discount_active_pct,type_3_discount_active_pct,type_4_discount_active_pct,type_5_discount_active_pct,type_6_discount_active_pct,type_0_discount_avg_when_active,type_1_discount_avg_when_active,type_2_discount_avg_when_active,type_3_discount_avg_when_active,type_4_discount_avg_when_active,type_5_discount_avg_when_active,type_6_discount_avg_when_active
4352,4384,Chicken_83,Meat and fish,226,14.180177,10.619469,0.0,0.884956,0.0,0.000000,0.0,75.221239,0.136014,NaN,0.153695,NaN,NaN,NaN,0.180892
3781,3809,Beef_22,Meat and fish,798,17.786917,6.390977,0.0,0.000000,0.0,3.884712,0.0,72.932331,0.129618,NaN,NaN,NaN,0.100003,NaN,0.144755
3701,3729,Pork_25,Meat and fish,1129,62.487839,2.834367,0.0,0.000000,0.0,0.000000,0.0,72.010629,0.194622,NaN,NaN,NaN,NaN,NaN,0.176369
1899,1915,Chicken_37,Meat and fish,375,2220.599600,0.000000,0.0,0.000000,0.0,0.000000,0.0,69.066667,NaN,NaN,NaN,NaN,NaN,NaN,0.114946
3704,3732,Pork_25,Meat and fish,1129,85.103277,2.834367,0.0,0.000000,0.0,0.000000,0.0,68.733392,0.194283,NaN,NaN,NaN,NaN,NaN,0.169513
1930,1946,Chicken_41,Meat and fish,1373,148.342141,0.000000,0.0,0.000000,0.0,0.000000,0.0,68.463219,NaN,NaN,NaN,NaN,NaN,NaN,0.106133
3702,3730,Pork_25,Meat and fish,1130,94.100540,2.389381,0.0,0.000000,0.0,0.000000,0.0,67.168142,0.177276,NaN,NaN,NaN,NaN,NaN,0.199448
4694,4731,Chicken_90,Meat and fish,1331,79.557926,5.184072,0.0,0.000000,0.0,0.000000,0.0,67.167543,0.289873,NaN,NaN,NaN,NaN,NaN,0.143239
4353,4385,Chicken_83,Meat and fish,251,12.969323,5.179283,0.0,0.796813,0.0,0.000000,0.0,66.135458,0.314738,NaN,0.176785,NaN,NaN,NaN,0.171832
1929,1945,Chicken_41,Meat and fish,1363,245.388283,0.000000,0.0,0.000000,0.0,0.000000,0.0,64.856933,NaN,NaN,NaN,NaN,NaN,NaN,0.111153


In [27]:

active_discount_cols = [
    f"{col}_active_pct" for col in discount_cols
]

item_discount_summary["ever_discounted"] = (
    item_discount_summary[active_discount_cols].max(axis=1) > 0
)

discounted_items_by_category = (
    item_discount_summary
    .groupby("L1_category_name_en")
    .agg(
        total_items=("unique_id", "nunique"),
        items_ever_discounted=("ever_discounted", "sum")
    )
)

discounted_items_by_category["percent_items_ever_discounted"] = (
    discounted_items_by_category["items_ever_discounted"]
    / discounted_items_by_category["total_items"]
    * 100
).round(2)

display(discounted_items_by_category)


,total_items,items_ever_discounted,percent_items_ever_discounted
L1_category_name_en,,,
Bakery,1611,1250,77.59
Fruit and vegetable,3246,2766,85.21
Meat and fish,533,529,99.25


In [28]:
discount_cols = [f"type_{i}_discount" for i in range(7)]

discount_values = model_data[discount_cols].fillna(0).clip(lower=0)

model_data["active_discount_count"] = (
    discount_values.gt(0).sum(axis=1)
)

print("Number of active discount types per row:")
display(
    model_data["active_discount_count"]
    .value_counts()
    .sort_index()
    .rename_axis("active_discount_types")
    .to_frame("number_of_rows")
)

print("\nRows with 2 or more active discount types:")
display(
    model_data.loc[
        model_data["active_discount_count"] >= 2,
        ["unique_id", "name", "date", "sell_price_main"]
        + discount_cols
    ].head(20)
)


Number of active discount types per row:


,number_of_rows
active_discount_types,
0,3421932
1,565773
2,19588
3,125
4,1



Rows with 2 or more active discount types:


,unique_id,name,date,sell_price_main,type_0_discount,type_1_discount,type_2_discount,type_3_discount,type_4_discount,type_5_discount,type_6_discount
3,4845,Croissant_35,2023-04-29,646.26,0.20024,0.0,0.0,0.0,0.15312,0.0,0.0
25,4845,Croissant_35,2023-05-09,646.26,0.07080,0.0,0.0,0.0,0.15312,0.0,0.0
40,4845,Croissant_35,2023-04-30,646.26,0.20024,0.0,0.0,0.0,0.15312,0.0,0.0
106,4845,Croissant_35,2023-05-19,646.26,0.20024,0.0,0.0,0.0,0.15312,0.0,0.0
125,4845,Croissant_35,2023-05-01,646.26,0.20024,0.0,0.0,0.0,0.15312,0.0,0.0
218,4845,Croissant_35,2023-05-12,646.26,0.20024,0.0,0.0,0.0,0.15312,0.0,0.0
256,4845,Croissant_35,2023-05-22,646.26,0.20024,0.0,0.0,0.0,0.15312,0.0,0.0
278,4845,Croissant_35,2023-05-18,646.26,0.20024,0.0,0.0,0.0,0.15312,0.0,0.0
325,4845,Croissant_35,2023-05-16,646.26,0.20024,0.0,0.0,0.0,0.15312,0.0,0.0
357,4845,Croissant_35,2023-05-06,646.26,0.20024,0.0,0.0,0.0,0.15312,0.0,0.0


In [29]:

import pandas as pd

discount_cols = [f"type_{i}_discount" for i in range(7)]

# Use a separate view; preserve the original discount columns
d = model_data[discount_cols].fillna(0).clip(lower=0)

# Which discount types are positive on the same row?
model_data["active_discount_count"] = d.gt(0).sum(axis=1)

print("Rows by number of simultaneously active discount types:")
display(
    model_data["active_discount_count"]
    .value_counts()
    .sort_index()
    .rename_axis("active_types")
    .to_frame("row_count")
)

# Show the most common exact combinations
combination = d.gt(0).apply(
    lambda row: " + ".join(
        col for col in discount_cols if row[col]
    ) or "No active discount",
    axis=1
)

print("\nMost common combinations:")
display(combination.value_counts().head(15).to_frame("row_count"))

# Inspect actual rows with 2 or more active discount types
cols_to_show = [
    "unique_id", "name", "date", "warehouse",
    "sell_price_main", "sales"
] + discount_cols

print("\nExamples with multiple active discount types:")
display(
    model_data.loc[
        model_data["active_discount_count"] >= 2,
        cols_to_show
    ].head(15)
)


Rows by number of simultaneously active discount types:


,row_count
active_types,
0,3421932
1,565773
2,19588
3,125
4,1



Most common combinations:


,row_count
No active discount,3421932
type_6_discount,262305
type_4_discount,138375
type_0_discount,135968
type_2_discount,18657
type_4_discount + type_6_discount,9540
type_5_discount,7879
type_0_discount + type_6_discount,6459
type_1_discount,2467
type_0_discount + type_4_discount,2088



Examples with multiple active discount types:


,unique_id,name,date,warehouse,sell_price_main,sales,type_0_discount,type_1_discount,type_2_discount,type_3_discount,type_4_discount,type_5_discount,type_6_discount
3,4845,Croissant_35,2023-04-29,Budapest_1,646.26,34.52,0.20024,0.0,0.0,0.0,0.15312,0.0,0.0
25,4845,Croissant_35,2023-05-09,Budapest_1,646.26,28.18,0.07080,0.0,0.0,0.0,0.15312,0.0,0.0
40,4845,Croissant_35,2023-04-30,Budapest_1,646.26,40.15,0.20024,0.0,0.0,0.0,0.15312,0.0,0.0
106,4845,Croissant_35,2023-05-19,Budapest_1,646.26,33.93,0.20024,0.0,0.0,0.0,0.15312,0.0,0.0
125,4845,Croissant_35,2023-05-01,Budapest_1,646.26,31.25,0.20024,0.0,0.0,0.0,0.15312,0.0,0.0
218,4845,Croissant_35,2023-05-12,Budapest_1,646.26,56.21,0.20024,0.0,0.0,0.0,0.15312,0.0,0.0
256,4845,Croissant_35,2023-05-22,Budapest_1,646.26,66.49,0.20024,0.0,0.0,0.0,0.15312,0.0,0.0
278,4845,Croissant_35,2023-05-18,Budapest_1,646.26,48.15,0.20024,0.0,0.0,0.0,0.15312,0.0,0.0
325,4845,Croissant_35,2023-05-16,Budapest_1,646.26,35.89,0.20024,0.0,0.0,0.0,0.15312,0.0,0.0
357,4845,Croissant_35,2023-05-06,Budapest_1,646.26,31.62,0.20024,0.0,0.0,0.0,0.15312,0.0,0.0


In [30]:
print(display(sales_with_inventory))

,unique_id,date,warehouse,total_orders,sales,sell_price_main,availability,type_0_discount,type_1_discount,type_2_discount,type_3_discount,type_4_discount,type_5_discount,type_6_discount,product_unique_id,name,L1_category_name_en,L2_category_name_en,L3_category_name_en,L4_category_name_en
0,4845,2024-03-10,Budapest_1,6436.0,16.34,646.26,1.00,0.00000,0.0,0.0,0.0,0.15312,0.0,0.0,2375,Croissant_35,Bakery,Bakery_L2_18,Bakery_L3_83,Bakery_L4_1
1,4845,2021-05-25,Budapest_1,4663.0,12.63,455.96,1.00,0.00000,0.0,0.0,0.0,0.15025,0.0,0.0,2375,Croissant_35,Bakery,Bakery_L2_18,Bakery_L3_83,Bakery_L4_1
2,4845,2021-12-20,Budapest_1,6507.0,34.55,455.96,1.00,0.00000,0.0,0.0,0.0,0.15025,0.0,0.0,2375,Croissant_35,Bakery,Bakery_L2_18,Bakery_L3_83,Bakery_L4_1
3,4845,2023-04-29,Budapest_1,5463.0,34.52,646.26,0.96,0.20024,0.0,0.0,0.0,0.15312,0.0,0.0,2375,Croissant_35,Bakery,Bakery_L2_18,Bakery_L3_83,Bakery_L4_1
4,4845,2022-04-01,Budapest_1,5997.0,35.92,486.41,1.00,0.00000,0.0,0.0,0.0,0.15649,0.0,0.0,2375,Croissant_35,Bakery,Bakery_L2_18,Bakery_L3_83,Bakery_L4_1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4007414,4941,2023-06-21,Prague_1,9988.0,26.56,34.06,1.00,0.00000,0.0,0.0,0.0,0.00000,0.0,0.0,2422,Kohlrabi_9,Fruit and vegetable,Fruit and vegetable_L2_3,Fruit and vegetable_L3_114,Fruit and vegetable_L4_1
4007415,4941,2023-06-24,Prague_1,8518.0,27.42,34.06,1.00,0.00000,0.0,0.0,0.0,0.00000,0.0,0.0,2422,Kohlrabi_9,Fruit and vegetable,Fruit and vegetable_L2_3,Fruit and vegetable_L3_114,Fruit and vegetable_L4_1
4007416,4941,2023-06-23,Prague_1,10424.0,33.39,34.06,1.00,0.00000,0.0,0.0,0.0,0.00000,0.0,0.0,2422,Kohlrabi_9,Fruit and vegetable,Fruit and vegetable_L2_3,Fruit and vegetable_L3_114,Fruit and vegetable_L4_1
4007417,4941,2023-06-22,Prague_1,10342.0,22.88,34.06,1.00,0.00000,0.0,0.0,0.0,0.00000,0.0,0.0,2422,Kohlrabi_9,Fruit and vegetable,Fruit and vegetable_L2_3,Fruit and vegetable_L3_114,Fruit and vegetable_L4_1


None


In [31]:

print("Availability summary:")
display(model_data["availability"].describe())

print("\nMost common values:")
display(
    model_data["availability"]
    .value_counts(dropna=False)
    .head(15)
    .to_frame("row_count")
)


Availability summary:


count    4.007419e+06
mean     9.253107e-01
std      1.796205e-01
min      1.000000e-02
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: availability, dtype: float64


Most common values:


,row_count
availability,
1.00,3035211
0.97,40419
0.98,39833
0.96,36086
0.94,26253
0.95,25048
0.93,23467
0.99,20844
0.92,19608
